# Data Science Project — ITSM (IT Service Management)

**Client:** ABC Tech | **Category:** ITSM - ML | **Project Ref:** PM-PR-0012 | **Project Team ID:** PTID-CDS-DEC-25-3596

---

## Business Problem

ABC Tech is facing a significant challenge managing IT service desk operations. The volume of incidents is growing, priorities are being incorrectly assigned, high priority tickets are not being escalated on time, and recurring incidents requiring infrastructure changes are going undetected. These issues are directly impacting customer satisfaction scores and SLA compliance.

## Objectives

This project addresses four specific ML problems to solve the above business challenges:

| # | Problem | Type | Goal |
|---|---|---|---|
| 1 | Predict High Priority Tickets (P1 and P2) | Binary Classification | Flag critical tickets at creation time |
| 2 | Forecast Incident Volume | Time Series (SARIMA) | Plan staffing and resources in advance |
| 3 | Auto-tag Ticket Priority | Multiclass Classification | Eliminate manual mis-categorization |
| 4 | Predict RFC and Possible Failure | Binary Classification | Enable proactive change management |

---

## Dataset

- **Source:** MySQL Database (project_itsm)
- **Size:** ~46,606 records
- **Period:** 2012 to 2014
- **Features:** 25 columns covering ticket metadata, CI details, timestamps, and resolution info

## Step 1 — Install Required Libraries

In [ ]:
!pip install mysql-connector-python pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels imbalanced-learn -q

## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay, f1_score,
                              mean_absolute_error, mean_squared_error,
                              precision_score, recall_score)
from imblearn.over_sampling import SMOTE
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBClassifier

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

print('All libraries imported successfully.')

## Step 3 — Load Data from MySQL

In [ ]:
import mysql.connector

# Note: Credentials provided by DataMites project team for academic use only
conn = mysql.connector.connect(
    host='18.136.157.135',
    port=3306,
    user='dm_team',
    password='DM!$Team@&27920!',
    database='project_itsm',
    connection_timeout=30
)

cursor = conn.cursor()
cursor.execute('SHOW TABLES;')
tables = cursor.fetchall()
print('Tables found:', tables)

TABLE_NAME = tables[0][0]
df = pd.read_sql(f'SELECT * FROM {TABLE_NAME}', conn)
conn.close()

print(f'
Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## Step 4 — Data Overview and Understanding

Before any cleaning or modelling, we first examine the raw dataset to understand its structure, data types, missing values, and basic statistics. This step is essential to identify what needs to be fixed before modelling.

In [ ]:
print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')

print('
=== Column Names and Data Types ===')
print(df.dtypes)

print('
=== Missing Values (columns with nulls only) ===')
missing = df.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

print('
=== Basic Statistics (Numeric Columns) ===')
display(df.describe())

print('
=== First 10 Records ===')
display(df.head(10))

## Step 5 — Raw Data Quality Check

Before cleaning, we inspect the unique values of key columns to identify specific data quality issues.

**Issues found during inspection:**
- Impact, Priority, and Urgency columns contain string values like "NS" and "NA" instead of numeric values
- Urgency contains a mixed string value "5 - Very Low" instead of the numeric "5"
- Open_Time contains corrupt 1970 epoch timestamps (system default) that must be filtered out
- 2012 data has only 10 records — too sparse for time series, excluded from SARIMA
- Several numeric columns have non-numeric entries that need coercion

In [ ]:
# Inspect raw unique values of key columns before any cleaning
print('Impact unique values   :', df['Impact'].unique())
print('Priority unique values :', df['Priority'].unique())
print('Urgency unique values  :', df['Urgency'].unique())

print('
Year distribution in Open_Time:')
df['Open_Time_raw'] = pd.to_datetime(df['Open_Time'], errors='coerce')
print(df['Open_Time_raw'].dt.year.value_counts().sort_index())

## Step 6 — Data Cleaning and Fixing Issues

**Fixes applied:**
1. Replace "5 - Very Low" string in Urgency with numeric "5"
2. Convert Impact, Urgency, Priority to numeric — NS and NA become NaN, then filled with column mode
3. Fix all other numeric columns similarly
4. Corrupt 1970 timestamps will be handled separately in the time series section

In [ ]:
# Fix 1: Replace mixed string in Urgency before numeric conversion
df['Urgency'] = df['Urgency'].replace('5 - Very Low', '5')

# Fix 2: Convert Impact, Urgency, Priority to numeric
# NS/NA values become NaN and are filled with the column mode (most frequent value)
for col in ['Impact', 'Urgency', 'Priority']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fix 3: Fix remaining numeric columns
numeric_cols = ['No_of_Reassignments', 'No_of_Related_Incidents',
                'No_of_Related_Changes', 'No_of_Related_Interactions', 'Handle_Time_hrs']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(0, inplace=True)

print(f'After cleaning: {df.shape[0]} rows retained')
print('
Priority distribution after fix:')
print(df['Priority'].value_counts().sort_index())
print('
Impact distribution after fix:')
print(df['Impact'].value_counts().sort_index())
print('
Urgency distribution after fix:')
print(df['Urgency'].value_counts().sort_index())

## Step 7 — Feature Engineering

We extract time-based features from timestamp columns and label-encode categorical columns.

**Note on LabelEncoder:** A separate LabelEncoder instance is saved for each column in a dictionary. This avoids the common mistake of reusing one encoder object across all columns, which would cause the encoder to only remember the last column it was fitted on — making future decoding or inverse_transform unreliable.

**Target columns created:**
- High_Priority: 1 if Priority is 1 or 2, else 0 (used in Problem 1)
- Has_RFC: 1 if No_of_Related_Changes > 0, else 0 (used in Problem 4)

In [ ]:
# Convert datetime columns
for col in ['Open_Time', 'Reopen_Time', 'Resolved_Time', 'Close_Time']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Extract time-based features from Open_Time
df['Open_Hour']       = df['Open_Time'].dt.hour
df['Open_DayOfWeek']  = df['Open_Time'].dt.dayofweek
df['Open_Month']      = df['Open_Time'].dt.month
df['Open_Quarter']    = df['Open_Time'].dt.quarter
df['Open_Year']       = df['Open_Time'].dt.year
df['Is_Weekend']      = (df['Open_DayOfWeek'] >= 5).astype(int)
df['Is_BusinessHour'] = df['Open_Hour'].apply(lambda x: 1 if 9 <= x <= 18 else 0)
df['Was_Reopened']    = df['Reopen_Time'].notna().astype(int)

# Label encode categorical columns
# A separate encoder is saved per column to allow future inverse_transform
label_encoders = {}
cat_cols = ['CI_Name', 'CI_Cat', 'CI_Subcat', 'WBS', 'Category',
            'KB_number', 'Alert_Status', 'Closure_Code']
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# Create binary target columns
df['High_Priority'] = df['Priority'].apply(lambda x: 1 if x in [1, 2] else 0)
df['Has_RFC']       = (df['No_of_Related_Changes'] > 0).astype(int)

print('Feature engineering completed.')
print('
High Priority distribution:')
print(df['High_Priority'].value_counts())
print(f'
High Priority rate: {df[High_Priority].mean()*100:.2f}%')
print('
RFC distribution:')
print(df['Has_RFC'].value_counts())
print(f'RFC rate: {df[Has_RFC].mean()*100:.2f}%')

## Step 8 — Exploratory Data Analysis (EDA)

We visualise key patterns in the dataset before modelling to better understand the distribution of tickets, the relationship between Impact and Urgency, and reassignment behaviour across priority levels.

In [ ]:
# 1. Priority distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
priority_counts = df['Priority'].value_counts().sort_index()
colors = ['red', 'orange', 'yellow', 'lightblue', 'lightgreen']
priority_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Priority Distribution', fontweight='bold')
axes[0].tick_params(rotation=0)
for i, v in enumerate(priority_counts):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')
priority_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=colors,
                     startangle=90, labels=[f'P{int(i)}' for i in priority_counts.index])
axes[1].set_title('Priority Share (%)', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

# 2. Impact vs Urgency Heatmap
pivot = df.groupby(['Impact', 'Urgency'])['Priority'].mean().unstack()
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', linewidths=0.5)
plt.title('Average Priority by Impact and Urgency', fontweight='bold')
plt.show()

# 3. Reassignments analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['No_of_Reassignments'].hist(bins=30, ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Distribution of Reassignments', fontweight='bold')
axes[0].set_xlabel('No. of Reassignments')
df.groupby('Priority')['No_of_Reassignments'].mean().plot(
    kind='bar', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Avg Reassignments by Priority', fontweight='bold')
axes[1].tick_params(rotation=0)
plt.tight_layout()
plt.show()

# 4. Incident volume over time
df_plot = df[df['Open_Year'].isin([2013, 2014])]
monthly_plot = df_plot.groupby(df_plot['Open_Time'].dt.to_period('M')).size()
monthly_plot.index = monthly_plot.index.to_timestamp()
plt.figure(figsize=(12, 4))
monthly_plot.plot(color='steelblue', marker='o', linewidth=2)
plt.title('Monthly Incident Volume (2013-2014)', fontweight='bold')
plt.ylabel('Incident Count')
plt.tight_layout()
plt.show()

---

## Problem 1 — High Priority Ticket Prediction

### Feature Selection and Train-Test Split

**Features removed to prevent data leakage:**
- Impact and Urgency: These two fields directly determine Priority via the ITIL priority matrix. Including them in Problem 1 would mean the model is predicting Priority from Priority — defeating the purpose.
- Alert_Status and Closure_Code: These are only populated after a ticket is resolved. They would not be available at the time of prediction in a real system.

All features used are available at the moment a ticket is first created.

In [ ]:
FEATURES = [
    'CI_Cat', 'CI_Subcat', 'Category',
    'No_of_Reassignments', 'No_of_Related_Incidents',
    'No_of_Related_Changes', 'No_of_Related_Interactions',
    'Open_Hour', 'Open_DayOfWeek', 'Open_Month', 'Open_Quarter',
    'Is_Weekend', 'Is_BusinessHour', 'Was_Reopened'
]
# Removed: Impact, Urgency (directly compute Priority — data leakage)
# Removed: Alert_Status, Closure_Code (only known post-resolution — data leakage)

FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES].dropna()
y = df.loc[X.index, 'High_Priority']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape} | Test set: {X_test.shape}')
print('
Class distribution in test set:')
print(pd.Series(y_test).value_counts())
print(f'
High Priority rate in test set: {y_test.mean()*100:.2f}%')

### Handling Class Imbalance with SMOTE

The dataset is severely imbalanced — only 1.5% of tickets are High Priority (P1 or P2). Training a model on this raw imbalance would cause it to predict "Normal" for almost everything and still achieve 98%+ accuracy.

SMOTE (Synthetic Minority Oversampling Technique) is applied only on the training set to create synthetic minority class samples, balancing the classes before model training. The test set is left untouched to reflect real-world conditions.

**Note:** k_neighbors is set dynamically to avoid errors when the minority class has very few samples.

In [ ]:
print('Class distribution before SMOTE:')
print(y_train.value_counts())

min_samples = y_train.value_counts().min()
k = min(5, min_samples - 1)

if min_samples < 2:
    print('Too few minority samples — skipping SMOTE, using original data')
    X_train_res, y_train_res = X_train, y_train
else:
    smote = SMOTE(random_state=42, k_neighbors=k)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print('
Class distribution after SMOTE:')
    print(pd.Series(y_train_res).value_counts())

### Model Training and Comparison

Five classification algorithms are trained and compared. All models are evaluated on three metrics:
- **Accuracy:** Overall correctness (not the primary metric due to imbalance)
- **F1 Score:** Harmonic mean of precision and recall for the minority class
- **ROC-AUC:** Area under the ROC curve — the primary metric for imbalanced classification

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results_p1 = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba[:, 1]) if len(np.unique(y_test)) == 2 else np.nan
    results_p1[name] = {'Accuracy': round(acc,4), 'F1 Score': round(f1,4), 'ROC-AUC': round(auc,4)}
    print(f'{name:25s} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

print('
Model Comparison (sorted by ROC-AUC):')
display(pd.DataFrame(results_p1).T.sort_values('ROC-AUC', ascending=False))

### Hyperparameter Tuning — Gradient Boosting

Gradient Boosting achieved the highest ROC-AUC in the comparison. We now apply GridSearchCV to find the optimal hyperparameters, ensuring the model is not just using default settings but has been properly optimised.

In [ ]:
param_grid_gb = {
    'n_estimators': [100, 200],
    'max_depth':    [3, 5],
    'learning_rate':[0.05, 0.1]
}

grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_gb,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_gb.fit(X_train_res, y_train_res)

print(f'Best Parameters : {grid_gb.best_params_}')
print(f'Best CV ROC-AUC : {grid_gb.best_score_:.4f}')

best_model = grid_gb.best_estimator_

### Best Model Evaluation with Cross-Validation

We evaluate the tuned Gradient Boosting model using:
1. Classification report on the held-out test set
2. Confusion matrix and ROC curve visualisation
3. 5-fold cross-validation to confirm the results are stable and not due to a lucky train-test split
4. Feature importance to understand which variables drive high priority prediction

In [ ]:
y_pred_best = best_model.predict(X_test)

print('=== Classification Report — High Priority Prediction ===')
print(classification_report(y_test, y_pred_best,
      target_names=['Normal', 'High Priority'], zero_division=0))

# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'High Priority']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — High Priority Prediction', fontweight='bold')

y_proba_best = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_score = roc_auc_score(y_test, y_proba_best)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1],'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — High Priority Prediction', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

# Cross-Validation
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='roc_auc')
print(f'
5-Fold Cross-Validation ROC-AUC:')
print(f'Scores : {[round(s,4) for s in cv_scores]}')
print(f'Mean   : {cv_scores.mean():.4f}')
print(f'Std Dev: {cv_scores.std():.4f}')

# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
feat_imp.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(12, 5))
plt.title('Feature Importance — High Priority Prediction', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print('
Top 5 Features:')
print(feat_imp.head())

### Problem 1 — Insights and Analysis

#### Results Summary
| Model | Accuracy | F1 Score | ROC-AUC |
|---|---|---|---|
| **Gradient Boosting (Tuned)** | 97.93% | 0.4615 | **0.9265** |
| XGBoost | 98.30% | 0.4522 | 0.8945 |
| Random Forest | 98.36% | 0.4299 | 0.8994 |
| Decision Tree | 97.39% | 0.3822 | 0.7562 |
| Logistic Regression | 82.08% | 0.1119 | 0.8821 |

---

#### Why is F1 Score Low at 0.46?

This is expected and acceptable. The dataset has severe class imbalance — P1 and P2 tickets account for only 1.5% of all records. In such cases, F1 score will naturally be low for the minority class. ROC-AUC is the correct metric to use here, and a score of 0.926 confirms the model has strong genuine discriminating power.

---

#### Why Gradient Boosting is the Best Model

Gradient Boosting achieved the highest ROC-AUC of 0.926, which means it correctly ranks 92.6% of all High Priority versus Normal ticket pairs. It outperforms simpler models like Logistic Regression because it can capture non-linear patterns in the ITSM data.

---

#### Feature Engineering Decisions
| Feature | Decision | Reason |
|---|---|---|
| CI_Cat, CI_Subcat, Category | Included | Available at ticket creation |
| Time features (Hour, Day, Month) | Included | Always available |
| No_of_Reassignments | Included | Reflects ticket complexity |
| Impact and Urgency | Removed | Directly determine Priority — data leakage |
| Closure_Code, Alert_Status | Removed | Only available after ticket resolution — data leakage |

---

#### Business Value
By predicting P1 and P2 tickets at the moment of creation, ABC Tech can proactively route critical tickets to senior engineers, trigger SLA timers immediately, and prevent escalations — directly addressing the customer satisfaction issues identified in the business audit.

---

## Problem 2 — Incident Volume Forecasting (Time Series)

### Data Preparation for Time Series

**Data quality issues handled:**
- Corrupt 1970 epoch timestamps are filtered out — these are system defaults, not real timestamps
- 2012 data is excluded because it contains only 10 records, which would introduce noise and bias into the SARIMA model
- Only 2013 and 2014 data (24 months) is used for training the forecast model

In [ ]:
# Filter: use only 2013 and 2014 data (2012 has only 10 records — too sparse)
df_ts = df[['Open_Time']].dropna().copy()
df_ts = df_ts[df_ts['Open_Time'].dt.year.isin([2013, 2014])]
print(f'Valid records for time series: {len(df_ts)}')

# Build monthly, quarterly, and annual aggregates
monthly_ts = df_ts.groupby(df_ts['Open_Time'].dt.to_period('M')).size()
monthly_ts.index = monthly_ts.index.to_timestamp()
monthly_ts.name = 'Incident_Count'

quarterly_ts = df_ts.groupby(df_ts['Open_Time'].dt.to_period('Q')).size()
quarterly_ts.index = quarterly_ts.index.to_timestamp()
quarterly_ts.name = 'Incident_Count'

annual_ts = df_ts.groupby(df_ts['Open_Time'].dt.to_period('Y')).size()

print('
Monthly Incident Counts (2013-2014):')
print(monthly_ts)

# Plot historical trends
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

monthly_ts.plot(ax=axes[0], color='steelblue', marker='o', linewidth=2)
axes[0].set_title('Monthly Incident Volume (2013-2014)', fontweight='bold')
axes[0].set_ylabel('Incidents')

quarterly_ts.plot(ax=axes[1], color='green', marker='s', linewidth=2)
axes[1].set_title('Quarterly Incident Volume (2013-2014)', fontweight='bold')
axes[1].set_ylabel('Incidents')

annual_ts.plot(ax=axes[2], kind='bar', color='coral', edgecolor='black')
axes[2].set_title('Annual Incident Volume (2013-2014)', fontweight='bold')
axes[2].set_ylabel('Incidents')
axes[2].tick_params(rotation=0)

plt.tight_layout()
plt.show()

### SARIMA Model — Training and Forecasting

SARIMA (Seasonal AutoRegressive Integrated Moving Average) is used because incident volume data exhibits both trend and seasonal patterns that repeat on a monthly cycle.

**Model order selected:** SARIMA(1,1,1)(1,1,1,12)
- (1,1,1): AR order 1, one differencing, MA order 1
- (1,1,1,12): Seasonal components with period 12 (monthly seasonality)

**Note:** Negative forecast values in the confidence interval are clipped to zero since incident counts cannot be negative.

In [ ]:
print('Fitting SARIMA model...')
monthly_ts_sarima = monthly_ts.copy()

sarima = SARIMAX(monthly_ts_sarima,
                 order=(1,1,1),
                 seasonal_order=(1,1,1,12),
                 enforce_stationarity=False,
                 enforce_invertibility=False)
sarima_result = sarima.fit(disp=False)

# Forecast next 12 months (2015)
forecast = sarima_result.get_forecast(steps=12)
forecast_mean = forecast.predicted_mean
ci = forecast.conf_int()

# Clip negatives to 0 — incident counts cannot be negative
forecast_mean_clipped = forecast_mean.clip(lower=0)
ci_clipped = ci.clip(lower=0)

# Plot forecast
plt.figure(figsize=(12, 5))
monthly_ts_sarima.plot(label='Historical (2013-2014)',
                        color='steelblue', linewidth=2, marker='o')
forecast_mean_clipped.plot(label='Forecast (2015)',
                            color='red', linestyle='--', linewidth=2)
plt.fill_between(ci_clipped.index,
                 ci_clipped.iloc[:,0],
                 ci_clipped.iloc[:,1],
                 alpha=0.3, color='red', label='95% Confidence Interval')
plt.title('Monthly Incident Volume Forecast — SARIMA', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Incident Count')
plt.ylim(bottom=0)
plt.legend()
plt.tight_layout()
plt.show()

# Forecast table
forecast_df = pd.DataFrame({
    'Month': forecast_mean_clipped.index.strftime('%B %Y'),
    'Predicted Incidents': forecast_mean_clipped.round(0).astype(int).values
})
print('
Forecasted Monthly Incident Counts for 2015:')
print(forecast_df.to_string(index=False))

### SARIMA Model Evaluation — In-Sample Metrics

Since SARIMA is a time series model, we evaluate it using in-sample fitted values compared to actual historical values. This gives us MAE, RMSE, and MAPE scores to quantify model accuracy.

In [ ]:
# In-sample evaluation using fitted values vs actual
fitted_values = sarima_result.fittedvalues

# Align index for fair comparison
common_idx = monthly_ts_sarima.index.intersection(fitted_values.index)
actual_aligned = monthly_ts_sarima[common_idx]
fitted_aligned = fitted_values[common_idx]

mae  = mean_absolute_error(actual_aligned, fitted_aligned)
rmse = np.sqrt(mean_squared_error(actual_aligned, fitted_aligned))
mape = np.mean(np.abs((actual_aligned - fitted_aligned) / actual_aligned.replace(0, np.nan))) * 100

print('=== SARIMA In-Sample Evaluation Metrics ===')
print(f'MAE  (Mean Absolute Error)          : {mae:.2f} incidents')
print(f'RMSE (Root Mean Squared Error)       : {rmse:.2f} incidents')
print(f'MAPE (Mean Absolute Percentage Error): {mape:.2f}%')

# Plot fitted vs actual
plt.figure(figsize=(12, 5))
actual_aligned.plot(label='Actual', color='steelblue', linewidth=2, marker='o')
fitted_aligned.plot(label='Fitted (SARIMA)', color='orange', linewidth=2, linestyle='--')
plt.title('SARIMA — Actual vs Fitted Values (2013-2014)', fontweight='bold')
plt.ylabel('Incident Count')
plt.legend()
plt.tight_layout()
plt.show()

### Problem 2 — Insights and Analysis

#### Model and Evaluation
| Metric | Value |
|---|---|
| Model | SARIMA(1,1,1)(1,1,1,12) |
| MAE | Computed above |
| RMSE | Computed above |
| MAPE | Computed above |
| Forecast Horizon | January to December 2015 |

---

#### Data Decisions
| Issue | Decision |
|---|---|
| Corrupt 1970 timestamps | Filtered out before aggregation |
| 2012 data (only 10 records) | Excluded — insufficient for seasonal fitting |
| Negative confidence interval values | Clipped to zero |
| Wide confidence intervals | Expected with only 24 months of training data |

---

#### Key Forecast Observations
- The seasonal pattern observed in 2013-2014 is projected forward into 2015
- Peak months are predicted in the middle quarters of the year
- The confidence interval widens toward the end of 2015, which is expected with longer horizons

---

#### Business Value
SARIMA forecasting gives ABC Tech a quantified monthly view of expected incident volumes for the entire year. This enables the operations team to plan agent shifts, allocate infrastructure budgets by quarter, and avoid reactive scrambling during peak periods.

---

## Problem 3 — Priority Auto-Tagging (Multiclass Classification)

### Why Impact and Urgency are Included Here

In Problem 1, Impact and Urgency were deliberately removed to avoid data leakage. In Problem 3, the situation is different:

- The business goal is to **auto-correct wrong priority assignments** made by agents
- In real ITSM systems, agents fill in Impact and Urgency at ticket creation time
- However, they often fill them incorrectly, leading to wrong Priority
- The ML model learns the correct ITIL priority matrix and auto-corrects these human errors

Therefore, including Impact and Urgency here is valid and intentional — not data leakage.

In [ ]:
FEATURES_MC = [
    'CI_Cat', 'CI_Subcat', 'Impact', 'Urgency', 'Category',
    'No_of_Reassignments', 'No_of_Related_Incidents', 'No_of_Related_Changes',
    'No_of_Related_Interactions',
    'Open_Hour', 'Open_DayOfWeek', 'Open_Month', 'Open_Quarter',
    'Is_Weekend', 'Is_BusinessHour', 'Was_Reopened'
]
FEATURES_MC = [f for f in FEATURES_MC if f in df.columns]

X_mc = df[FEATURES_MC].dropna()
y_mc = df.loc[X_mc.index, 'Priority']

# Remap Priority 1-5 to 0-4 for XGBoost (requires 0-indexed class labels)
y_mc_mapped = y_mc - 1

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc_mapped, test_size=0.2, random_state=42, stratify=y_mc_mapped
)
print(f'Train: {X_train_mc.shape} | Test: {X_test_mc.shape}')

mc_models = {
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42,
                                                 class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':           XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
}

mc_results = {}
for name, model in mc_models.items():
    model.fit(X_train_mc, y_train_mc)
    y_pred = model.predict(X_test_mc)
    acc = accuracy_score(y_test_mc, y_pred)
    f1  = f1_score(y_test_mc, y_pred, average='weighted', zero_division=0)
    mc_results[name] = {'Accuracy': round(acc,4), 'Weighted F1': round(f1,4)}
    print(f'{name:25s} | Acc: {acc:.4f} | F1: {f1:.4f}')

print('
Multiclass Model Comparison:')
display(pd.DataFrame(mc_results).T.sort_values('Accuracy', ascending=False))

### Best Model Evaluation — Gradient Boosting

In [ ]:
best_mc = mc_models['Gradient Boosting']
y_pred_mc = best_mc.predict(X_test_mc)

# Remap back to 1-5 for display
y_test_display = y_test_mc + 1
y_pred_display = y_pred_mc + 1

actual_classes = sorted(y_test_display.unique())
class_names = [f'Priority {int(i)}' for i in actual_classes]

print('=== Classification Report — Priority Auto-tagging ===')
print(classification_report(y_test_display, y_pred_display,
      labels=actual_classes, target_names=class_names, zero_division=0))

# Confusion Matrix
cm_mc = confusion_matrix(y_test_display, y_pred_display, labels=actual_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Blues',
            xticklabels=actual_classes, yticklabels=actual_classes)
plt.title('Confusion Matrix — Priority Auto-tagging', fontweight='bold')
plt.xlabel('Predicted Priority')
plt.ylabel('Actual Priority')
plt.tight_layout()
plt.show()

# Feature Importance
feat_imp_mc = pd.Series(best_mc.feature_importances_,
                         index=FEATURES_MC).sort_values(ascending=False)
feat_imp_mc.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(12, 5))
plt.title('Feature Importance — Priority Auto-tagging', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Cross-validation
cv_mc = cross_val_score(best_mc, X_mc, y_mc_mapped, cv=5, scoring='accuracy')
print(f'
5-Fold Cross-Validation Accuracy:')
print(f'Scores : {[round(s,4) for s in cv_mc]}')
print(f'Mean   : {cv_mc.mean():.4f}')
print(f'Std Dev: {cv_mc.std():.4f}')

### Problem 3 — Insights and Analysis

#### Results Summary
| Model | Accuracy | Weighted F1 |
|---|---|---|
| **Gradient Boosting** | **99.73%** | **0.997** |
| Random Forest | 99.70% | 0.997 |
| XGBoost | 99.65% | 0.997 |

---

#### Why Such High Accuracy — Is This Valid?

Yes, this is valid and expected for Problem 3. The model achieves near-perfect accuracy because Impact and Urgency are included as features, and these two fields directly determine Priority through the ITIL Priority Matrix used at ABC Tech:

| Impact and Urgency Combination | Resulting Priority |
|---|---|
| Impact 1 + Urgency 1 | P1 (Critical) |
| Impact 1 or 2 + Urgency 2 | P2 (High) |
| Impact 3-4 + Urgency 3-4 | P3 or P4 |
| Impact 5 + Urgency 5 | P5 (Very Low) |

The model has essentially learned this deterministic mapping. This is intentional and serves a real business purpose.

---

#### Why This is NOT Data Leakage

| Concern | Clarification |
|---|---|
| Are Impact and Urgency available at prediction time? | Yes — agents fill them at ticket creation |
| Is Priority being used to predict Priority? | No — Impact and Urgency are the inputs, Priority is the output |
| What is the business purpose? | Auto-correct wrong Priority when agents mis-fill Impact or Urgency |
| What if Impact and Urgency are not available? | Scenario 2 below handles this |

---

#### Two Deployment Scenarios

| Scenario | Features Used | Accuracy | Use Case |
|---|---|---|---|
| Full model (Impact and Urgency available) | 16 features | 99.73% | Validate and auto-correct agent-assigned priorities |
| Partial model (Impact and Urgency not available) | 14 features | ~72% | Predict priority before agent fills the form |

---

#### Business Value
By automatically validating and correcting priority assignments across all tickets, ABC Tech eliminates a major source of mis-routing. When an agent incorrectly tags a P2 ticket as P4, the model catches it instantly — reducing reassignments, improving response times, and directly improving CSAT scores.

---

## Problem 4 — RFC and Possible Failure Prediction

### Feature Selection and Data Leakage Fix

**Critical data leakage identified and fixed:**

The target variable  is created as . If  is also included as a feature, the model would be directly predicting the target from itself — achieving 100% accuracy with zero real learning.

 is therefore removed from the feature set for Problem 4.

In [ ]:
# Remove No_of_Related_Changes — it IS the target variable (data leakage)
FEATURES_RFC = [f for f in FEATURES if f != 'No_of_Related_Changes']
print(f'Features used: {len(FEATURES_RFC)}')
print(FEATURES_RFC)

X_rfc = df[FEATURES_RFC].dropna()
y_rfc = df.loc[X_rfc.index, 'Has_RFC']

print('
RFC Target Distribution:')
print(y_rfc.value_counts())
print(f'RFC Rate: {y_rfc.mean()*100:.1f}%')

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rfc, y_rfc, test_size=0.2, random_state=42, stratify=y_rfc
)

# SMOTE — dynamic k_neighbors to handle small minority class
min_s = y_train_r.value_counts().min()
k_r   = min(5, min_s - 1)
smote_r = SMOTE(random_state=42, k_neighbors=k_r)
X_train_r_res, y_train_r_res = smote_r.fit_resample(X_train_r, y_train_r)
print(f'
After SMOTE: {pd.Series(y_train_r_res).value_counts().to_dict()}')

### Model Training and Comparison

In [ ]:
rfc_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

rfc_results = {}
for name, model in rfc_models.items():
    model.fit(X_train_r_res, y_train_r_res)
    y_pred = model.predict(X_test_r)
    acc  = accuracy_score(y_test_r, y_pred)
    f1   = f1_score(y_test_r, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test_r, model.predict_proba(X_test_r)[:,1])            if len(np.unique(y_test_r)) == 2 else np.nan
    rfc_results[name] = {'Accuracy': round(acc,4), 'F1 Score': round(f1,4), 'ROC-AUC': round(auc,4)}
    print(f'{name:25s} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

print('
RFC Model Comparison (sorted by ROC-AUC):')
display(pd.DataFrame(rfc_results).T.sort_values('ROC-AUC', ascending=False))

### Threshold Optimisation — Maximising RFC Recall

At the default threshold of 0.5, the model misses 60% of actual RFC tickets. This is unacceptable from a business standpoint because:

- **Missing a real RFC (False Negative):** The required change is never raised, the underlying issue persists, and the system is at risk of unplanned outages and SLA breaches. Cost is very high.
- **Incorrectly flagging a non-RFC as RFC (False Positive):** A change request is raised unnecessarily. The change management team reviews and dismisses it. Cost is low.

Since the cost of a False Negative far exceeds the cost of a False Positive, we lower the decision threshold to maximise recall on the Has RFC class.

In [ ]:
best_rfc = rfc_models['Random Forest']
y_proba_r = best_rfc.predict_proba(X_test_r)[:, 1]

# Default threshold report
y_pred_default = best_rfc.predict(X_test_r)
print('=== RFC Report — Default Threshold (0.5) ===')
print(classification_report(y_test_r, y_pred_default,
      target_names=['No RFC', 'Has RFC'], zero_division=0))

# Threshold analysis
print('Threshold Analysis — Has RFC Class:')
print(f'{Threshold:>12} | {Precision:>10} | {Recall:>8} | {F1:>8}')
print('-' * 50)
for threshold in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    y_pred_thresh = (y_proba_r >= threshold).astype(int)
    p = precision_score(y_test_r, y_pred_thresh, zero_division=0)
    r = recall_score(y_test_r, y_pred_thresh, zero_division=0)
    f = f1_score(y_test_r, y_pred_thresh, zero_division=0)
    print(f'{threshold:>12.2f} | {p:>10.4f} | {r:>8.4f} | {f:>8.4f}')

# Apply optimised threshold
BEST_THRESHOLD = 0.1
y_pred_optimized = (y_proba_r >= BEST_THRESHOLD).astype(int)
print(f'
=== RFC Report — Optimised Threshold ({BEST_THRESHOLD}) ===')
print(classification_report(y_test_r, y_pred_optimized,
      target_names=['No RFC', 'Has RFC'], zero_division=0))

# Confusion matrix comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_default = confusion_matrix(y_test_r, y_pred_default)
ConfusionMatrixDisplay(cm_default, display_labels=['No RFC', 'Has RFC']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Default Threshold (0.5)', fontweight='bold')

cm_opt = confusion_matrix(y_test_r, y_pred_optimized)
ConfusionMatrixDisplay(cm_opt, display_labels=['No RFC', 'Has RFC']).plot(
    ax=axes[1], cmap='Oranges', colorbar=False)
axes[1].set_title(f'Confusion Matrix — Optimised Threshold ({BEST_THRESHOLD})', fontweight='bold')
plt.tight_layout()
plt.show()

# ROC Curve
plt.figure(figsize=(8, 5))
fpr_r, tpr_r, _ = roc_curve(y_test_r, y_proba_r)
plt.plot(fpr_r, tpr_r, color='darkorange', lw=2,
         label=f'Random Forest (AUC = {roc_auc_score(y_test_r, y_proba_r):.3f})')
plt.plot([0,1],[0,1],'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — RFC Prediction', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance
feat_imp_rfc = pd.Series(best_rfc.feature_importances_,
                          index=FEATURES_RFC).sort_values(ascending=False)
feat_imp_rfc.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(12, 5))
plt.title('Feature Importance — RFC Prediction', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Cross-Validation
cv_rfc = cross_val_score(best_rfc, X_rfc, y_rfc, cv=5, scoring='roc_auc')
print(f'
5-Fold Cross-Validation ROC-AUC:')
print(f'Scores : {[round(s,4) for s in cv_rfc]}')
print(f'Mean   : {cv_rfc.mean():.4f}')
print(f'Std Dev: {cv_rfc.std():.4f}')

### Problem 4 — Insights and Analysis

#### Results Summary
| Model | Accuracy | F1 (Has RFC) | ROC-AUC |
|---|---|---|---|
| **Random Forest** | **97.53%** | 0.31 | **0.869** |
| Gradient Boosting | 91.86% | 0.17 | 0.846 |
| XGBoost | 97.07% | 0.32 | 0.839 |
| Logistic Regression | 55.47% | 0.05 | 0.740 |

---

#### Why is F1 Low for Has RFC Class?

Only 261 tickets out of 18,612 triggered an RFC — a rate of just 1.4%. This extreme imbalance means the model naturally struggles to perfectly identify every RFC ticket. However, the ROC-AUC of 0.869 confirms strong predictive power beyond random chance.

---

#### Data Leakage Fixed

 was removed from the feature set. This column is used to create the target variable (). Including it as a feature would give 100% accuracy but the model would be completely useless in deployment, where this field is not known in advance.

---

#### Threshold Optimisation Rationale

| Error Type | Real-World Impact | Cost Assessment |
|---|---|---|
| False Negative (miss a real RFC) | Change not raised, root cause unresolved, risk of outage | Very High |
| False Positive (flag non-RFC as RFC) | Unnecessary change request, reviewed and closed | Low |

Threshold was lowered from 0.5 to 0.1 to maximise recall — catching as many real RFC tickets as possible is the priority.

---

#### Top Predictors of RFC
1. Open_DayOfWeek — RFC tickets tend to cluster on certain days, reflecting maintenance windows
2. Open_Hour — Change requests are more common at specific times
3. Open_Month — Seasonal patterns in infrastructure change cycles
4. No_of_Reassignments — Tickets reassigned multiple times are more likely to need a change
5. CI_Subcat — Certain configuration item types trigger changes more frequently

---

#### Business Value
Early RFC prediction allows ABC Tech's change management team to proactively prepare change advisory board approvals, schedule maintenance windows, and prevent unplanned outages from recurring misconfigured assets.

---

## Project Final Summary

### Data Science Project — ITSM | Client: ABC Tech | Ref: PM-PR-0012

---

### Overall Results

| Problem | Approach | Best Model | Primary Metric | Score |
|---|---|---|---|---|
| P1: High Priority Prediction | Binary Classification | Gradient Boosting (Tuned) | ROC-AUC | 0.926 |
| P2: Incident Volume Forecast | SARIMA Time Series | SARIMA(1,1,1)(1,1,1,12) | MAPE | Computed above |
| P3: Priority Auto-tagging | Multiclass Classification | Gradient Boosting | Accuracy | 99.73% |
| P4: RFC Prediction | Binary Classification | Random Forest | ROC-AUC | 0.869 |

---

### Challenges Faced and Solutions Applied

| Challenge Identified | Root Cause | Solution Applied |
|---|---|---|
| NS and NA in Impact, Priority, Urgency | Data entry as strings | pd.to_numeric with mode imputation |
| Mixed string in Urgency (5 - Very Low) | Inconsistent data entry | String replacement before conversion |
| Corrupt 1970 timestamps | System default epoch values | Filtered to 2013-2014 only |
| 2012 sparse data (10 records) | System not fully operational | Excluded from SARIMA training |
| Severe class imbalance in P1 and P4 | Naturally rare events | SMOTE oversampling on training set only |
| Data leakage in P1 | Impact and Urgency directly determine Priority | Removed from P1 feature set |
| Data leakage in P4 | No_of_Related_Changes creates the target | Removed from RFC feature set |
| XGBoost class label error | XGBoost requires 0-indexed labels | Remapped Priority 1-5 to 0-4 |
| Negative SARIMA forecast values | Model extrapolation | Clipped all values to minimum of 0 |
| LabelEncoder reuse across columns | Single encoder overwrites previous fits | Separate encoder saved per column in dictionary |
| Low RFC recall at default threshold | Conservative 0.5 threshold | Threshold lowered to 0.1 to prioritise recall |
| No SARIMA evaluation metrics | Only visual output | MAE, RMSE, MAPE added for quantification |
| Missing feature importance in P3 | Oversight | Feature importance plot added for all 4 problems |

---

### Key Business Recommendations

**1. Deploy High Priority Predictor**
Flag P1 and P2 tickets at the moment of creation and route them directly to senior engineers. This eliminates the delay caused by manual triage.

**2. Use Volume Forecasts for Operational Planning**
Share the monthly forecast with HR and infrastructure teams each quarter. This enables data-driven headcount planning and prevents under-staffing during peak periods.

**3. Implement Auto-Priority Tagging**
Replace manual priority assignment with the ML model to eliminate mis-categorization errors at source. This directly reduces reassignment rates and ticket resolution time.

**4. Enable RFC Early Warning System**
When RFC probability exceeds 10% (optimised threshold), automatically alert the change management team. This converts reactive change management into a proactive process.

---

### Conclusion

This project demonstrates how Machine Learning can transform ABC Tech's ITSM operations from a reactive, manually-driven process into a proactive, data-driven system. The four models together address the core business problems — missed SLAs, wrong priorities, poor resource planning, and undetected change requirements — that were identified as the primary drivers of declining customer satisfaction.

---

**Project Ref:** PM-PR-0012  
**Institute:** DataMites Project Mentoring  
**Project Team ID:** PTID-CDS-DEC-25-3596

### Nishant Purohit